# Laboratorium 4 (4 pkt.)

Celem czwartego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmów głębokiego uczenia aktywnego. Zaimplementowane algorytmy będą testowane z wykorzystaniem wcześniej przygotowanych środowisk: *FrozenLake* i *Pacman* oraz środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gym
import numpy as np
import random

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Dołączenie bibliotek ze środowiskami:

In [2]:
from env.FrozenLakeMDP import frozenLake
from env.FrozenLakeMDPExtended import frozenLakeExtended


Dołączenie bibliotek do obsługi sieci neuronowych

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(342)

class DQN(nn.Module):
    def __init__(self, state_size, action_size, hidden_neurons, learning_rate):
        super(DQN, self).__init__()

        self.fc1 = nn.Linear(state_size, hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons)
        self.out = nn.Linear(hidden_neurons, action_size)

        self.learning_rate = learning_rate
        self.optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        return self.out(x)
    
    def predict(self, state):
        state = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.numpy()
    
    def fit(self, states, targets):
        if len(states) == 0:
            return
        
        states = np.array(states)
        targets = np.array(targets)
        
        if states.ndim == 1:
            states = states.reshape(1, -1)
        if targets.ndim == 1:
            targets = targets.reshape(1, -1)

        states = torch.FloatTensor(states)
        targets = torch.FloatTensor(targets)

        self.optimizer.zero_grad()
        outputs = self.forward(states)
        loss = F.smooth_l1_loss(outputs, targets)
        loss.backward()
        self.optimizer.step()

## Zadanie 1 - Deep Q-Network

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Deep Q-Network. Wartoscią oczekiwaną sieci jest:
\begin{equation}
        Q(s_t, a_t) = r_{t+1} + \gamma \text{max}_a Q(s_{t + 1}, a)
\end{equation}
</p>

In [ ]:
np.random.seed(342)

class DQNAgent:
    def __init__(self, action_size, learning_rate, model):
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95    # discount rate
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.95
        self.learning_rate = learning_rate
        self.model = model

    def remember(self, state, action, reward, next_state, done):
        #Function adds information to the memory about last action and its results
        self.memory.append((state, action, reward, next_state, done)) 

    def get_action(self, state):
        """
        Compute the action to take in the current state, including exploration.
        With probability self.epsilon, we should take a random action.
            otherwise - the best policy action (self.get_best_action).

        Note: To pick randomly from a list, use random.choice(list).
              To pick True or False with a given probablity, generate uniform number in [0, 1]
              and compare it with your probability
        """

        if random.uniform(0, 1) < self.epsilon:
            chosen_action = random.choice(range(self.action_size))
        else:
            chosen_action = self.get_best_action(state)

        
        return chosen_action

  
    def get_best_action(self, state):
        """
        Compute the best action to take in a state.
        """
        q_values = self.model.predict(state) 
        
        best_value = np.max(q_values)
        best_actions = np.where(q_values == best_value)[0]
        best_action = random.choice(best_actions)

        return best_action

    def replay(self, batch_size):
        """
        Function learn network using randomly selected actions from the memory. 
        First calculates Q value for the next state and choose action with the biggest value.
        Target value is calculated according to:
                Q(s,a) := (r + gamma * max_a(Q(s', a)))
        except the situation when the next action is the last action, in such case Q(s, a) := r.
        In order to change only those weights responsible for chosing given action, the rest values should be those
        returned by the network for state state.
        The network should be trained on batch_size samples.
        """
        if len(self.memory) == 0:
            return
        
        used_batch_size = 0
        if len(self.memory) < batch_size:
            used_batch_size = len(self.memory)

        mini_batch = random.sample(self.memory, used_batch_size)

        states = []
        targets = []

        for state, action, reward, next_state, done in mini_batch:
            target = reward
            if not done:
                target = reward + self.gamma * np.amax(self.model.predict(next_state))
            target_f = self.model.predict(state)
            target_f[action] = target
            
            states.append(state)
            targets.append(target_f)

        self.model.fit(states, targets)

    def update_epsilon_value(self):
        #Every each epoch epsilon value should be updated according to equation: 
        #self.epsilon *= self.epsilon_decay, but the updated value shouldn't be lower then epsilon_min value
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

Czas przygotować model sieci, która będzie się uczyła poruszania po środowisku *FrozenLake*, warstwa wejściowa powinna mieć tyle neuronów ile jest możlliwych stanów, warstwa wyjściowa tyle neuronów ile jest możliwych akcji do wykonania:

In [5]:
env = frozenLake("8x8")

state_size = env.get_number_of_states()
action_size = len(env.get_possible_actions(None))
learning_rate = 0.005

model = DQN(state_size, action_size, 16, learning_rate)

 Czas nauczyć agenta poruszania się po środowisku *FrozenLake*, jako stan przyjmij wektor o liczbie elementów równej liczbie możliwych stanów, z wartością 1 ustawioną w komórce o indeksie równym aktualnemu stanowi, pozostałe elementy mają być wypełnione zerami:
* 1 pkt < 35 epok,
* 0.5 pkt < 60 epok,
* 0.25 pkt - w pozostałych przypadkach.

In [6]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75

done = False
batch_size = 64
EPISODES = 10000
counter = 0
for e in range(EPISODES):

    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()

    
        state = np.zeros(state_size)
        state[env_state] = 1
        
        for time in range(1000):
            action = agent.get_action(state)
            next_state_env, reward, done, _ = env.step(action)
            total_reward += reward

            next_state = np.zeros(state_size) 
            next_state[next_state_env] = 1

            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break
        
        agent.replay(batch_size)
        summary.append(total_reward)
    
    agent.update_epsilon_value()

    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))
    if np.mean(summary) > 0.9:
        print ("You Win!")
        break

epoch #0	mean reward = 0.010	epsilon = 0.712
epoch #1	mean reward = 0.000	epsilon = 0.677
epoch #2	mean reward = 0.000	epsilon = 0.643
epoch #3	mean reward = 0.000	epsilon = 0.611
epoch #4	mean reward = 0.000	epsilon = 0.580
epoch #5	mean reward = 0.010	epsilon = 0.551
epoch #6	mean reward = 0.000	epsilon = 0.524
epoch #7	mean reward = 0.010	epsilon = 0.498
epoch #8	mean reward = 0.020	epsilon = 0.473
epoch #9	mean reward = 0.090	epsilon = 0.449
epoch #10	mean reward = 0.120	epsilon = 0.427
epoch #11	mean reward = 0.080	epsilon = 0.405
epoch #12	mean reward = 0.290	epsilon = 0.385
epoch #13	mean reward = 0.190	epsilon = 0.366
epoch #14	mean reward = 0.160	epsilon = 0.347
epoch #15	mean reward = 0.270	epsilon = 0.330
epoch #16	mean reward = 0.500	epsilon = 0.314
epoch #17	mean reward = 0.470	epsilon = 0.298
epoch #18	mean reward = 0.580	epsilon = 0.283
epoch #19	mean reward = 0.630	epsilon = 0.269
epoch #20	mean reward = 0.790	epsilon = 0.255
epoch #21	mean reward = 0.730	epsilon = 0.24

Czas przygotować model sieci, która będzie się uczyła poruszania po środowisku *FrozenLakeExtended*, tym razem stan nie jest określany poprzez pojedynczą liczbę, a przez 3 tablice:
* pierwsza zawierająca informacje o celu,
* druga zawierająca informacje o dziurach,
* trzecia zawierająca informację o położeniu gracza.

In [7]:
env = frozenLakeExtended("4x4")

state_size = env.get_number_of_states()
action_size = len(env.get_possible_actions(None))
learning_rate = 0.005

model = DQN(state_size * 3, action_size, 32, learning_rate)

 Czas nauczyć agenta poruszania się po środowisku *FrozenLakeExtended*, jako stan przyjmij wektor składający się ze wszystkich trzech tablic (2 pkt.):

In [8]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75

done = False
batch_size = 64
EPISODES = 2000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()

        state = [np.asarray(part).ravel() for part in env_state]
        state = np.concatenate(state).astype(float)
        
        for time in range(1000):
            action = agent.get_action(state)
            next_state_env, reward, done, _ = env.step(action)
            total_reward += reward

            next_state = [np.asarray(part).ravel() for part in next_state_env]
            next_state = np.concatenate(next_state).astype(float)

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

        agent.replay(batch_size)
        summary.append(total_reward)
    
    agent.update_epsilon_value()

    if np.mean(summary) > 0.9:
        print ("You Win!")
        break
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))

epoch #0	mean reward = 0.000	epsilon = 0.712
epoch #1	mean reward = 0.030	epsilon = 0.677
epoch #2	mean reward = 0.020	epsilon = 0.643
epoch #3	mean reward = 0.010	epsilon = 0.611
epoch #4	mean reward = 0.190	epsilon = 0.580
epoch #5	mean reward = 0.200	epsilon = 0.551
epoch #6	mean reward = 0.090	epsilon = 0.524
epoch #7	mean reward = 0.190	epsilon = 0.498
epoch #8	mean reward = 0.130	epsilon = 0.473
epoch #9	mean reward = 0.180	epsilon = 0.449
epoch #10	mean reward = 0.240	epsilon = 0.427
epoch #11	mean reward = 0.070	epsilon = 0.405
epoch #12	mean reward = 0.260	epsilon = 0.385
epoch #13	mean reward = 0.060	epsilon = 0.366
epoch #14	mean reward = 0.280	epsilon = 0.347
epoch #15	mean reward = 0.350	epsilon = 0.330
epoch #16	mean reward = 0.090	epsilon = 0.314
epoch #17	mean reward = 0.470	epsilon = 0.298
epoch #18	mean reward = 0.260	epsilon = 0.283
epoch #19	mean reward = 0.490	epsilon = 0.269
epoch #20	mean reward = 0.270	epsilon = 0.255
epoch #21	mean reward = 0.470	epsilon = 0.24

Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [9]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.005

print("State size: {}, Action size: {}".format(state_size, action_size))

model = DQN(state_size, action_size, 2, learning_rate)

State size: 4, Action size: 2


c:\Users\Filip\Documents\mgr-siium\G_uczenie_ze_wzmocnieniem\.venv\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


Czas nauczyć agenta gry w środowisku *CartPool*:
* 1 pkt < 10 epok,
* 0.5 pkt < 20 epok,
* 0.25 pkt - w pozostałych przypadkach.

In [10]:
agent = DQNAgent(action_size, learning_rate, model)

agent.epsilon = 0.75

done = False
batch_size = 64
EPISODES = 1000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()

        state = np.asarray(env_state[0]).ravel().astype(float)

        for time in range(300):
            action = agent.get_action(state)
            if isinstance(action, np.ndarray):
                action = int(action[0])
            
            if not hasattr(np, "bool8"):
                np.bool8 = np.bool_
            step_value = env.step(int(action))
            if len(step_value) == 4:
                next_state_env, reward, done, _ = step_value
            else:
                next_state_env, reward, done, _, _ = step_value
            total_reward += reward

            next_state = np.asarray(next_state_env).ravel().astype(float)

            #add to experience memory
            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

        agent.replay(batch_size)
        summary.append(total_reward)


    agent.update_epsilon_value()

    if np.mean(summary) > 195:
        print ("You Win!")
        break
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))

epoch #0	mean reward = 17.530	epsilon = 0.712
epoch #1	mean reward = 17.480	epsilon = 0.677
epoch #2	mean reward = 16.520	epsilon = 0.643
epoch #3	mean reward = 16.210	epsilon = 0.611
epoch #4	mean reward = 16.180	epsilon = 0.580
epoch #5	mean reward = 16.220	epsilon = 0.551
epoch #6	mean reward = 14.740	epsilon = 0.524
epoch #7	mean reward = 13.220	epsilon = 0.498
epoch #8	mean reward = 13.990	epsilon = 0.473
epoch #9	mean reward = 13.540	epsilon = 0.449
epoch #10	mean reward = 13.150	epsilon = 0.427
epoch #11	mean reward = 12.160	epsilon = 0.405
epoch #12	mean reward = 12.320	epsilon = 0.385
epoch #13	mean reward = 12.150	epsilon = 0.366
epoch #14	mean reward = 11.950	epsilon = 0.347
epoch #15	mean reward = 11.920	epsilon = 0.330
epoch #16	mean reward = 11.660	epsilon = 0.314
epoch #17	mean reward = 11.430	epsilon = 0.298
epoch #18	mean reward = 11.070	epsilon = 0.283
epoch #19	mean reward = 11.750	epsilon = 0.269
epoch #20	mean reward = 10.880	epsilon = 0.255
epoch #21	mean reward =

KeyboardInterrupt: 